# Financial Transactions Analytics Dashboard

This notebook analyses a synthetic financial transactions dataset using Python and pandas. The goal is to identify income, expenses, monthly cash flow, recurring payments, unusual activity, category spend, merchant spend and budget performance.

## 1. Load Libraries and Data

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "cleaned" / "transactions_cleaned.csv"
df = pd.read_csv(DATA_PATH, parse_dates=["transaction_date", "transaction_month"])
df.head()

## 2. Data Quality Checks

These checks confirm the dataset has loaded correctly and is ready for analysis.

In [ ]:
quality_summary = pd.DataFrame({
    "rows": [len(df)],
    "columns": [df.shape[1]],
    "duplicate_transaction_ids": [df["transaction_id"].duplicated().sum()],
    "missing_values": [df.isna().sum().sum()],
    "first_transaction_date": [df["transaction_date"].min().date()],
    "last_transaction_date": [df["transaction_date"].max().date()],
})

quality_summary

In [ ]:
df.info()

## 3. Summary Statistics

In [ ]:
df[["amount", "amount_abs", "income_amount", "expense_amount", "net_amount", "balance_after_transaction"]].describe().round(2)

## 4. Key Financial Metrics

In [ ]:
total_income = df["income_amount"].sum()
total_expenses = df["expense_amount"].sum()
net_cash_flow = df["net_amount"].sum()
transaction_count = len(df)
unusual_count = df["is_unusual"].sum()

kpi_summary = pd.DataFrame({
    "Metric": ["Total Income", "Total Expenses", "Net Cash Flow", "Transaction Count", "Unusual Transactions"],
    "Value": [total_income, total_expenses, net_cash_flow, transaction_count, unusual_count],
})

kpi_summary

## 5. Monthly Income, Expenses and Net Cash Flow

In [ ]:
monthly_cash_flow = (
    df.groupby("transaction_month")
    .agg(
        total_income=("income_amount", "sum"),
        total_expenses=("expense_amount", "sum"),
        net_cash_flow=("net_amount", "sum"),
        transaction_count=("transaction_id", "count"),
    )
    .reset_index()
)

monthly_cash_flow

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(monthly_cash_flow["transaction_month"], monthly_cash_flow["total_income"], marker="o", label="Income")
ax.plot(monthly_cash_flow["transaction_month"], monthly_cash_flow["total_expenses"], marker="o", label="Expenses")
ax.bar(monthly_cash_flow["transaction_month"], monthly_cash_flow["net_cash_flow"], alpha=0.25, label="Net Cash Flow")
ax.set_title("Monthly Income, Expenses and Net Cash Flow")
ax.set_xlabel("Month")
ax.set_ylabel("Amount")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Spending by Category

In [ ]:
category_spend = (
    df[df["transaction_type"] == "Expense"]
    .groupby("category")
    .agg(total_spend=("expense_amount", "sum"), transactions=("transaction_id", "count"))
    .sort_values("total_spend", ascending=False)
)

category_spend

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
category_spend["total_spend"].sort_values().plot(kind="barh", ax=ax)
ax.set_title("Total Spending by Category")
ax.set_xlabel("Spend")
ax.set_ylabel("Category")
plt.tight_layout()
plt.show()

## 7. Spending by Merchant

In [ ]:
merchant_spend = (
    df[df["transaction_type"] == "Expense"]
    .groupby(["merchant", "category"])
    .agg(total_spend=("expense_amount", "sum"), transactions=("transaction_id", "count"))
    .sort_values("total_spend", ascending=False)
    .head(15)
)

merchant_spend

## 8. Recurring Payments Analysis

In [ ]:
recurring_payments = (
    df[df["is_recurring"]]
    .groupby(["merchant", "category", "payment_method"])
    .agg(
        recurring_count=("transaction_id", "count"),
        recurring_total=("amount_abs", "sum"),
        average_value=("amount_abs", "mean"),
    )
    .sort_values("recurring_total", ascending=False)
)

recurring_payments.head(20)

## 9. Unusual Transaction Analysis

In [ ]:
unusual_transactions = df[df["is_unusual"]].sort_values("amount_abs", ascending=False)
unusual_transactions[["transaction_id", "transaction_date", "customer_id", "account_id", "category", "merchant", "amount", "payment_method"]].head(20)

## 10. Payment Method Breakdown

In [ ]:
payment_breakdown = (
    df.groupby("payment_method")
    .agg(transaction_count=("transaction_id", "count"), total_value=("amount_abs", "sum"))
    .sort_values("transaction_count", ascending=False)
)

payment_breakdown

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
payment_breakdown["transaction_count"].sort_values().plot(kind="barh", ax=ax)
ax.set_title("Transaction Count by Payment Method")
ax.set_xlabel("Transactions")
ax.set_ylabel("Payment Method")
plt.tight_layout()
plt.show()

## 11. Budget Performance Analysis

In [ ]:
budget_performance = (
    df[df["category_budget"] > 0]
    .drop_duplicates(["transaction_month", "category"])
    .groupby("category")
    .agg(
        average_monthly_spend=("category_month_expense", "mean"),
        monthly_budget=("category_budget", "max"),
        over_budget_months=("budget_status", lambda x: (x == "Over Budget").sum()),
    )
    .assign(average_variance=lambda x: x["monthly_budget"] - x["average_monthly_spend"])
    .sort_values("average_monthly_spend", ascending=False)
)

budget_performance.round(2)

## 12. Final Insights

- Compare income and expenses monthly to identify positive or negative cash flow trends.
- Review high recurring payments because they can become fixed-cost pressure over time.
- Investigate unusual transactions, especially high-value payments and duplicate-looking payments.
- Monitor categories that repeatedly exceed budget, then separate essential and discretionary spend.
- Use the dashboard plan in `dashboard/powerbi_dashboard_plan.md` to convert this analysis into a Power BI report.